# Demographic Dataset Processing

## Overview

This notebook processes a single demographic dataset obtained from the UK Census.

The objective is to clean, reorganise, and extract the required demographic information for the proposed framework.

The processed dataset will be exported as a standardised JSON file for later integration.

---

## Dataset Information

| Item | Description |
|------|-------------|
| Dataset |RM062_Industry_by_age |
| Source | Office for National Statistics (ONS) Census 2021 |
| Geography | local authorities: district / unitary (as of April 2023) |
| Input File |RM062_Industry_by_age |
| Output File |industry_by_age.json |
| Notebook Author | Peter |
| Last Updated |2026-08-21|

## 1. Import

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import json

In [2]:
import sys
from pathlib import Path

# Find repository root (../..)
repo_root = Path.cwd().parents[1]

# Add repository root to Python import path
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
# Display settings

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configuration

Specify the input dataset and output location.

In [4]:
# File paths

input_file = "RM062_Industry_by_age.xlsx"
output_file = "industry_by_age.json"

raw_data_path = Path("../../data/demographic/raw/industry")
output_data_path = Path("../../data/demographic/processed/industry/")

## 3. Load Dataset

In [5]:
df = pd.read_excel(raw_data_path / input_file)
df_processed = df.copy()

df_processed.head()

,Lower tier local authorities Code,Lower tier local authorities,Industry (current) (19 categories) Code,Industry (current) (19 categories),Age (86 categories) Code,Age (86 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0,Aged under 1 year,918
1,E06000001,Hartlepool,-8,Does not apply,1,Aged 1 year,954
2,E06000001,Hartlepool,-8,Does not apply,2,Aged 2 years,1046
3,E06000001,Hartlepool,-8,Does not apply,3,Aged 3 years,988
4,E06000001,Hartlepool,-8,Does not apply,4,Aged 4 years,1077


## 4. Dataset Exploration

Inspect the structure of the dataset before processing.

In [6]:
df_processed.info()

df_processed.shape

df_processed.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 524514 entries, 0 to 524513
Data columns (total 7 columns):
 #   Column                                   Non-Null Count   Dtype 
---  ------                                   --------------   ----- 
 0   Lower tier local authorities Code        524514 non-null  object
 1   Lower tier local authorities             524514 non-null  object
 2   Industry (current) (19 categories) Code  524514 non-null  int64 
 3   Industry (current) (19 categories)       524514 non-null  object
 4   Age (86 categories) Code                 524514 non-null  int64 
 5   Age (86 categories)                      524514 non-null  object
 6   Observation                              524514 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 28.0+ MB


Index(['Lower tier local authorities Code', 'Lower tier local authorities',
       'Industry (current) (19 categories) Code',
       'Industry (current) (19 categories)', 'Age (86 categories) Code',
       'Age (86 categories)', 'Observation'],
      dtype='object')

In [7]:
df_processed["Industry (current) (19 categories)"].unique()

array(['Does not apply', 'A Agriculture, forestry and fishing',
       'B Mining and quarrying', 'C Manufacturing',
       'D Electricity, gas, steam and air conditioning supply',
       'E Water supply; sewerage, waste management and remediation activities',
       'F Construction',
       'G Wholesale and retail trade; repair of motor vehicles and motor cycles',
       'H Transport and storage',
       'I Accommodation and food service activities',
       'J Information and communication',
       'K Financial and insurance activities', 'L Real estate activities',
       'M Professional, scientific and technical activities',
       'N Administrative and support service activities',
       'O Public administration and defence; compulsory social security',
       'P Education', 'Q Human health and social work activities',
       'R, S, T, U Other'], dtype=object)

In [8]:
df_processed.describe(include="all")

,Lower tier local authorities Code,Lower tier local authorities,Industry (current) (19 categories) Code,Industry (current) (19 categories),Age (86 categories) Code,Age (86 categories),Observation
count,524514,524514,524514.000000,524514,524514.000000,524514,524514.000000
unique,321,321,NaN,19,NaN,86,NaN
top,W06000024,Merthyr Tydfil,NaN,Does not apply,NaN,Aged under 1 year,NaN
freq,1634,1634,NaN,27606,NaN,6099,NaN
mean,NaN,NaN,8.578947,NaN,42.500000,NaN,111.917726
std,NaN,NaN,6.385152,NaN,24.824407,NaN,383.378211
min,NaN,NaN,-8.000000,NaN,0.000000,NaN,0.000000
25%,NaN,NaN,4.000000,NaN,21.000000,NaN,0.000000
50%,NaN,NaN,9.000000,NaN,42.500000,NaN,15.000000
75%,NaN,NaN,14.000000,NaN,64.000000,NaN,85.000000


## 5. Data Cleaning

Perform dataset-specific cleaning operations.

Typical tasks include:

- Rename variables
- Remove unnecessary columns
- Handle missing values
- Convert data types
- Standardise labels

In [9]:
# Cleaning code
selected_columns = ['Lower tier local authorities', 
                    'Industry (current) (19 categories)',
                    'Age (86 categories) Code',
                    'Observation']
df_processed = df_processed[selected_columns]

df_processed = df_processed.rename(columns={
    'Industry (current) (19 categories)': 'Industry',
    'Age (86 categories) Code': 'Age',
    'Observation': 'Industry Count'
})

## 6. Feature Processing

Extract the demographic information required by the framework.

Examples include:

- Age groups
- Household composition
- Occupation
- Industry
- Car ownership
- Student status

The processing steps in this section depend on the current dataset.

In [10]:
# Dataset-specific processing
from ontology.FRAMEWORK_AGE_GROUP import (
    FRAMEWORK_AGE_GROUP,
    AGE_ORDER
)

In [11]:
def get_industry_distribution(local_authority):
    filtered_data = df_processed[
        df_processed['Lower tier local authorities'] == local_authority].copy()

    filtered_data['Age'] = filtered_data['Age'].apply(FRAMEWORK_AGE_GROUP)

    result = (filtered_data.groupby(['Lower tier local authorities', 
                                     'Industry',
                                     'Age'],as_index=False)['Industry Count'].sum())

    result['Age'] = pd.Categorical(result['Age'], categories=AGE_ORDER, ordered=True)

    result = result.sort_values('Age').reset_index(drop=True)

    return result

In [12]:
from ontology.STUDY_AREA import STUDY_AREA

In [13]:
get_industry_distribution(STUDY_AREA)

,Lower tier local authorities,Industry,Age,Industry Count
0,Manchester,"A Agriculture, forestry and fishing",0-4,0
1,Manchester,F Construction,0-4,0
2,Manchester,O Public administration and defence; compulsor...,0-4,0
3,Manchester,N Administrative and support service activities,0-4,0
4,Manchester,H Transport and storage,0-4,0
...,...,...,...,...
166,Manchester,"E Water supply; sewerage, waste management and...",75+,0
167,Manchester,O Public administration and defence; compulsor...,75+,24
168,Manchester,J Information and communication,75+,4
169,Manchester,B Mining and quarrying,75+,0


## 7. Validation

Verify that the processed dataset is internally consistent before exporting.

In [14]:
# Examples

df_processed.head()

df_processed.info()

df_processed.isna().sum()

# Additional validation here

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 524514 entries, 0 to 524513
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Lower tier local authorities  524514 non-null  object
 1   Industry                      524514 non-null  object
 2   Age                           524514 non-null  int64 
 3   Industry Count                524514 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 16.0+ MB


Lower tier local authorities    0
Industry                        0
Age                             0
Industry Count                  0
dtype: int64

## 8. Export Processed Dataset

In [15]:
output_data_path.mkdir(parents=True, exist_ok=True)

# Example

# df_processed.to_json(
#     output_data_path / output_file,
#     orient="records",
#     indent=4
# )

industry_by_age_df = get_industry_distribution("Manchester")
industry_by_age_df.to_json(
    output_data_path / output_file,
    orient="records",
    indent=4
)

## Summary

### Input

-

### Output

-

### Processing Completed

- Dataset loaded
- Dataset cleaned
- Features extracted
- Validation completed
- JSON exported